In [1]:
!pip install langchain langchain-experimental langchain-community langchain-openai openai chromadb pypdf sentence_transformers gradio langchain-together

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/13

In [4]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [5]:
import getpass


from langchain_community.utilities import WikipediaAPIWrapper, SerpAPIWrapper
from langchain_community.tools import WikipediaQueryRun
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage


# =====================================================
# Step 0: API Keys
# =====================================================
os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter OpenRouter API Key: ")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
os.environ["SERPAPI_KEY"] = getpass.getpass("Enter SERPAPI API Key: ")
SERPAPI_KEY = os.environ["SERPAPI_KEY"]

/tmp/ipykernel_1246/1502353167.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import WikipediaAPIWrapper, SerpAPIWrapper
/tmp/ipykernel_1246/1502353167.py:6: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.tools.python.tool import PythonREPLTool


Enter OpenRouter API Key: ··········
Enter SERPAPI API Key: ··········


In [6]:
#---------------------------------------
# Initialise LLM
#---------------------------------------

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.3,
    max_tokens=500,
    openai_api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

In [7]:
from langchain_core.messages import HumanMessage
from IPython.display import display, Markdown

request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.

Not a mathematical puzzle, but more of a thought-provoking question
that requires intelligent insight.

Include in your question that the answer must be short.

Answer only with the question, no explanation.
"""

response = llm.invoke(
    [HumanMessage(content=request)]
)

question = response.content

display(Markdown(question))

If you could eliminate one human emotion from existence, which would it be and why?

Multiple LLM Providers

In [8]:
from langchain_openai import ChatOpenAI
from IPython.display import display, Markdown

competitors = []
answers = []


def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(f"### {model_name}\n\n{answer}"))

In [9]:
def ask_model(model_name, question):

    llm = ChatOpenAI(
        model=model_name,
        temperature=0,
        max_tokens=300,
        openai_api_key=OPENROUTER_API_KEY,
        base_url="https://openrouter.ai/api/v1"
    )

    response = llm.invoke(question)

    return response.content

In [14]:
models = [

    # OpenAI
    "openai/gpt-4o-mini",

    # Anthropic
    "anthropic/claude-sonnet-4",

    # Google
    "google/gemini-2.5-flash",

    # Deepseek
    "deepseek/deepseek-chat-v3",

    # Grok
    "x-ai/grok-4.3",

    # Groq hosted models
    "groq/llama-3.3-70b-versatile",

    # Meta
    "meta-llama/llama-3.3-70b-instruct",

]

In [15]:
for model_name in models:

    try:
        answer = ask_model(model_name, question)
        record(model_name, answer)

    except Exception as e:
        print(f"{model_name} failed")
        print(e)

### openai/gpt-4o-mini

Eliminating a human emotion is a complex proposition, as emotions play crucial roles in our experiences and interactions. However, if I had to choose one, it might be hatred. Hatred often leads to conflict, division, and suffering, both on personal and societal levels. By removing hatred, we could potentially foster greater understanding, compassion, and cooperation among individuals and communities, leading to a more harmonious world. That said, it's important to recognize that emotions, even negative ones, can serve as catalysts for change and growth, so the implications of such a choice would need to be carefully considered.

### anthropic/claude-sonnet-4

I'd probably choose spite - that particular flavor of malice where someone acts primarily to harm others, even at cost to themselves.

Unlike other challenging emotions, spite seems uniquely destructive without much redeeming value. Anger can motivate us to address injustice. Fear keeps us safe. Even hatred, while dangerous, sometimes reflects violated values worth protecting.

But spite? It's essentially weaponized pettiness - the urge to make others suffer just because we can, or because we're hurting. It fuels cycles of retaliation, turns minor conflicts into lasting feuds, and makes people choose harm over healing.

That said, I'd hesitate to actually eliminate any emotion entirely. Our emotional complexity, even the uncomfortable parts, might serve purposes we don't fully understand. Maybe spite occasionally prevents us from being taken advantage of, or signals when our dignity has been violated.

But if I had to pick one emotion that seems to create more suffering than insight, spite would be my choice. The world might be a bit kinder without that particular impulse to inflict unnecessary pain.

What's your take - is there an emotion you think we'd be better off without, or do you see value in keeping our full messy emotional range?

### google/gemini-2.5-flash

This is a fascinating and complex question! If I had to choose one human emotion to eliminate, I would choose **resentment**.

Here's why:

*   **It's a corrosive, non-productive emotion:** Resentment is a bitter indignation at having been treated unfairly. Unlike anger, which can be a powerful motivator for change and justice, resentment tends to fester. It's a backward-looking emotion that dwells on past wrongs without offering a clear path forward.
*   **It damages relationships:** Resentment erodes trust and intimacy. It can lead to passive-aggressive behavior, avoidance, and ultimately, the breakdown of personal and professional relationships. It's hard to truly connect with someone when you're holding onto a deep-seated resentment towards them.
*   **It harms the individual experiencing it:** Holding onto resentment is emotionally draining. It can lead to chronic stress, anxiety, and even physical health problems. It prevents individuals from moving on, forgiving (themselves or others), and finding peace. It keeps people trapped in a cycle of bitterness.
*   **It doesn't typically lead to positive action:** While anger can fuel activism or confrontation that leads to resolution, resentment often leads to inaction or destructive, indirect actions. It's more about punishing the self by holding onto the pain than it is about seeking a constructive solution.
*   **Other "negative" emotions have more utility:**
    *   **Ang

### deepseek/deepseek-chat-v3

This is a fascinating and complex question! If I had to choose one emotion to eliminate, it might be **chronic shame**—not the fleeting, constructive kind that helps us course-correct, but the deep, persistent shame that corrodes self-worth and fuels cycles of self-destruction. Here’s why:

1. **Toxic Impact**: Unlike guilt (which says, "I did something bad"), shame says, "I *am* bad." It’s linked to mental health struggles like depression, addiction, and social isolation.  
2. **Barrier to Growth**: Shame often silences people, preventing them from seeking help or connecting with others. It’s a roadblock to vulnerability and healing.  
3. **Misplaced Function**: While some emotions (like fear or anger) have clear survival purposes, chronic shame rarely serves a productive role. It’s more often a weapon of control (e.g., societal or cultural shaming) than a tool for growth.  

**Alternate contenders and why I’d keep them**:  
- **Jealousy**: Though painful, it can reveal unmet needs or values.  
- **Anger**: A vital signal of injustice and a motivator for change.  
- **Fear**: Protects us from danger, even if it sometimes overfunctions.  

**Counterargument**: One might argue that shame maintains social order, but I’d suggest that guilt (focusing on *actions*) could do this

### x-ai/grok-4.3

**Envy.**

It's the one emotion that seems to offer zero upside while actively poisoning everything else. Fear can keep you alive, anger can drive necessary change, sadness helps process loss and build empathy, even guilt can steer moral behavior. Envy, by contrast, is pure subtraction: it makes you resent someone else's good fortune without improving your own situation, often pushing people toward pettiness, sabotage, or self-sabotage.

Remove envy and you might still have competition, ambition, and desire, but they'd be directed outward at the goal rather than sideways at the person who already reached it. A lot of online outrage, workplace toxicity, and quiet personal misery would lose their fuel source. The world would probably get slightly less bitter and slightly more productive as a result.

groq/llama-3.3-70b-versatile failed
Error code: 400 - {'error': {'message': 'groq/llama-3.3-70b-versatile is not a valid model ID', 'code': 400}, 'user_id': 'user_3GtzpIITzvVyeAwRgiWN1LRtYIF'}


### meta-llama/llama-3.3-70b-instruct

After careful consideration, I would propose eliminating envy from human emotions. Here's why:

Envy is a complex and often destructive emotion that can lead to negative consequences for individuals and society as a whole. It's characterized by feelings of resentment, bitterness, and discontent towards others who possess something we desire, such as success, wealth, status, or relationships. Envy can manifest in various forms, including:

1. **Comparison and competition**: Envy fosters an unhealthy focus on comparing ourselves to others, leading to an endless cycle of competition and one-upmanship. This can result in an excessive emphasis on external validation, rather than self-improvement and personal growth.
2. **Resentment and bitterness**: Envy can breed resentment and bitterness towards others, which can damage relationships, friendships, and even lead to conflicts.
3. **Unhappiness and dissatisfaction**: Envy can make us feel unhappy and dissatisfied with our own lives, as we constantly focus on what we lack rather than what we have.
4. **Destructive behavior**: In extreme cases, envy can drive people to engage in destructive behavior, such as sabotage, gossip, or even violence, in an attempt to bring others down or acquire what they perceive as lacking.

By eliminating envy, I believe humanity could benefit in several ways:

1. **Increased empathy and kindness**: Without envy, people might be more inclined to celebrate others' successes and feel genuine happiness for their achievements.
2. **Improved mental health**:

In [13]:
messages = question


model_name = "anthropic/claude-sonnet-4"

answer = ask_model(model_name, messages)
record(model_name, answer)


model_name = "google/gemini-2.5-flash"

answer = ask_model(model_name, messages)
record(model_name, answer)


model_name = "x-ai/grok-4.3"

answer = ask_model(model_name, messages)
record(model_name, answer)

### anthropic/claude-sonnet-4

I'd probably choose spite - that particular flavor of malice where someone acts primarily to hurt others, even at cost to themselves.

Unlike other challenging emotions, spite seems uniquely destructive without much redeeming value. Anger can motivate us to address injustice. Fear keeps us safe. Even hatred, while dangerous, sometimes reflects violated values worth protecting.

But spite? It's essentially weaponized pettiness - the urge to make others suffer just because we can, or because we're hurting. It fuels cycles of retaliation, turns minor conflicts into lasting feuds, and often harms the spiteful person as much as their target.

That said, I'd hesitate to actually eliminate any emotion entirely. Our emotional complexity, even the uncomfortable parts, seems deeply connected to what makes us human. Maybe spite occasionally serves some evolutionary purpose I'm not seeing, or emerges from the same neural pathways that enable more positive traits.

But if I had to pick one emotion that seems to cause disproportionate suffering relative to any benefit... spite would be my choice.

What's your take? Is there an emotion you think we'd be better off without, or do you see value in keeping the full messy spectrum of human feeling?

### google/gemini-2.5-flash

This is a fascinating and challenging question! If I had to eliminate one human emotion, I would choose **resentment**.

Here's why:

*   **It's a destructive, lingering emotion:** Unlike anger, which can be a powerful motivator for change and has a more immediate, often cathartic release, resentment is a slow-burning, corrosive emotion. It festers, grows, and poisons relationships over time. It's often rooted in past hurts that haven't been addressed or forgiven.

*   **It hinders forgiveness and reconciliation:** Resentment is a major barrier to moving forward. It keeps people trapped in the past, unable to forgive themselves or others, and prevents the healing of rifts.

*   **It's self-punishing:** While the person who caused the initial hurt might be oblivious, the person holding onto resentment is the one who suffers most. It consumes mental and emotional energy, leading to stress, anxiety, and even physical health problems.

*   **It rarely leads to positive action:** Unlike sadness (which can lead to empathy and seeking comfort), fear (which can lead to caution and self-preservation), or even jealousy (which can sometimes motivate self-improvement), resentment rarely inspires constructive behavior. Instead, it often leads to passive aggression, bitterness, and a desire for revenge, which only perpetuates cycles of negativity.

*   **It's often a choice:** While initial hurt might be unavoidable,

### x-ai/grok-4.3

**Envy.**

It's the one emotion that seems to offer zero upside while actively poisoning everything it touches. Fear can keep you alive, anger can drive necessary change, sadness helps process loss, even guilt can steer moral behavior. Envy, by contrast, is pure subtraction: it makes you resent someone else's win without improving your own situation, erodes relationships, fuels pointless status games, and often leads people to sabotage others (or themselves) rather than build anything worthwhile.

Eliminating it wouldn't remove ambition or the desire to improve—those can exist without the corrosive comparison layer. People would still want things; they just wouldn't feel poisoned by someone else already having them. The world would probably get slightly less bitter and slightly more collaborative as a result.

In [16]:
print(len(competitors))
print(competitors)
print(answers)

16
['openai/gpt-4o-mini', 'anthropic/claude-sonnet-4', 'google/gemini-2.5-flash', 'deepseek/deepseek-chat-v3', 'meta-llama/llama-3.3-70b-instruct', 'anthropic/claude-sonnet-4', 'google/gemini-2.5-flash', 'anthropic/claude-sonnet-4', 'google/gemini-2.5-flash', 'x-ai/grok-4.3', 'openai/gpt-4o-mini', 'anthropic/claude-sonnet-4', 'google/gemini-2.5-flash', 'deepseek/deepseek-chat-v3', 'x-ai/grok-4.3', 'meta-llama/llama-3.3-70b-instruct']
["Eliminating a human emotion is a complex proposition, as emotions play crucial roles in our experiences and interactions. However, if I had to choose one, it might be **hatred**. Hatred can lead to significant harm, conflict, and division among individuals and groups. It often fuels violence, discrimination, and suffering, creating cycles of negativity that can be difficult to break. By eliminating hatred, we could potentially foster greater understanding, compassion, and cooperation among people, leading to a more harmonious society. However, it's impor

In [17]:
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")

Competitor: openai/gpt-4o-mini

Eliminating a human emotion is a complex proposition, as emotions play crucial roles in our experiences and interactions. However, if I had to choose one, it might be **hatred**. Hatred can lead to significant harm, conflict, and division among individuals and groups. It often fuels violence, discrimination, and suffering, creating cycles of negativity that can be difficult to break. By eliminating hatred, we could potentially foster greater understanding, compassion, and cooperation among people, leading to a more harmonious society. However, it's important to recognize that emotions, even negative ones, can also serve as catalysts for change and self-reflection, so the implications of such a choice would need to be carefully considered.
Competitor: anthropic/claude-sonnet-4

I'd probably choose spite - that particular flavor of malice where someone acts primarily to hurt others, even at cost to themselves.

Unlike anger (which can motivate positive cha

In [18]:
together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [19]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""

In [20]:
print(judge)

You are judging a competition between 16 competitors.
Each model has been given this question:

If you could eliminate one human emotion from existence, which would it be and why?

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}

Here are the responses from each competitor:

# Response from competitor 1

Eliminating a human emotion is a complex proposition, as emotions play crucial roles in our experiences and interactions. However, if I had to choose one, it might be **hatred**. Hatred can lead to significant harm, conflict, and division among individuals and groups. It often fuels violence, discrimination, and suffering, creating cycles of negativity that can be difficult to break. By eliminating hatred, we could potentially foster greater understa

In [30]:
judge_messages = [{"role": "user", "content": judge}]
judge_llm = ChatOpenAI(
    model="google/gemini-2.5-pro",
    temperature=0,
    max_tokens=500,
    openai_api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

In [31]:
response = judge_llm.invoke(judge_messages)

print(response.content)

{
    "results":[3,1,4,2]
}


In [33]:
results_dict = json.loads(response.content)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

Rank 1: google/gemini-2.5-flash
Rank 2: openai/gpt-4o-mini
Rank 3: deepseek/deepseek-chat-v3
Rank 4: anthropic/claude-sonnet-4
